In [ ]:
"""version1"""
import xlwings as xw                            
import pdfplumber
import re
from datetime  import datetime,date 

files=['BKTS10720010091124053218.PDF','BKTS10720010091224045043.PDF','BKTS10740010091124170630.PDF','BKTS10750010091124131717.PDF','BKTS107100100911242219291.pdf','BKTS107100100911242221022.pdf','BKTS107100100912242237121.pdf','BKTS107100100912242238372.pdf','BKTS107300100911241636101.pdf','MT10790010091124165614.pdf']
files_series=['BKTS10720010091124053218.PDF','BKTS10720010091224045043.PDF','BKTS10750010091124131717.PDF','BKTS107100100911242219291.pdf','BKTS107100100911242221022.pdf','BKTS107100100912242237121.pdf','BKTS107100100912242238372.pdf']

today_lodgment_inward_data = {"branch_code": [],"branch_name": [],"item_count": [],"total_amount": [],"clearing_type": []}
today_lodgment_outward_data = {"branch_code": [],"branch_name": [],"item_count": [],"total_amount": [],"clearing_type": []}

clearing_inward_data = {"branch_code": [],"branch_name": [],"item_count": [],"total_amount": [],"clearing_type": []}
clearing_outward_data = {"branch_code": [],"branch_name": [],"item_count": [],"total_amount": [],"clearing_type": []}

return_inward_data = {"branch_code": [],"branch_name": [],"item_count": [],"total_amount": [],"clearing_type": []}
return_outward_data = {"branch_code": [],"branch_name": [],"item_count": [],"total_amount": [],"clearing_type": []}

for file in files:
  with pdfplumber.open(file) as pdf:
    text = "\n".join(page.extract_text() or "" for page in pdf.pages)

    inward = re.findall(r"RECEIVED BY YOU.*?(.*?)DELIVERED BY YOU", text,re.DOTALL)
    inward = ' '.join(inward)       #converting list to text or string
    
    branch_code_inw = re.findall(r"^\d{4}\s", inward,re.MULTILINE)
    branch_name_inw = re.findall(r"\d{4}\s+(.*?)\s+GROUP", inward,re.MULTILINE) 
    total_count_inw = re.findall(r"GROUP B\s*[:=]\s*(\d{1,2})\s",inward,re.MULTILINE)
    total_amount_inw = re.findall(r"GROUP B\s*[:=]\s\d{1,2}\s*([\d,.]+)", inward,re.MULTILINE)
    clearing_type_inw=["RECEIVED BY YOU"  for i in range(len(branch_code_inw)) ]

    outward = re.findall(r"DELIVERED BY YOU.*?(.*?)SUMMARY", text,re.DOTALL)
    outward = ' '.join(outward)

    branch_code_outw = re.findall(r"^\d{4}\s", outward,re.MULTILINE)
    branch_name_outw = re.findall(r"\d{4}\s+(.*?)\s+GROUP", outward,re.MULTILINE)
    total_count_outw = re.findall(r"GROUP B\s*[:=]\s*(\d{1,2})\s",outward,re.MULTILINE)
    total_amount_outw = re.findall(r"GROUP B\s*[:=]\s\d{1,2}\s*([\d,.]+)", outward,re.MULTILINE)
    clearing_type_outw=["DELIVERED BY YOU"   for i in range(len(branch_code_outw))]

    date_match=re.findall(r"CLEARING AS ON\s*(\d{2}-\d{2}-\d{4})",text,re.MULTILINE)

    valid_date=None 
                              #validating date as real date
    if date_match!=[]:
      valid_date = datetime.strptime(date_match[0], '%d-%m-%Y').date()
    else:
      pass


    if valid_date==date.today():            #matching date with today's date
        today_lodgment_inward_data["branch_code"].extend(branch_code_inw)
        today_lodgment_inward_data["branch_name"].extend(branch_name_inw)
        today_lodgment_inward_data["item_count"].extend(total_count_inw)
        today_lodgment_inward_data["total_amount"].extend(total_amount_inw)
        today_lodgment_inward_data["clearing_type"].extend(clearing_type_inw)
      
        today_lodgment_outward_data["branch_code"].extend(branch_code_outw)
        today_lodgment_outward_data["branch_name"].extend(branch_name_outw)
        today_lodgment_outward_data["item_count"].extend(total_count_outw)
        today_lodgment_outward_data["total_amount"].extend(total_amount_outw)
        today_lodgment_outward_data["clearing_type"].extend(clearing_type_outw)
    else:
       pass
  
    if file in files_series:
      
        clearing_inward_data["branch_code"].extend(branch_code_inw)
        clearing_inward_data["branch_name"].extend(branch_name_inw)
        clearing_inward_data["item_count"].extend(total_count_inw)
        clearing_inward_data["total_amount"].extend(total_amount_inw)
        clearing_inward_data["clearing_type"].extend(clearing_type_inw)
      
        clearing_outward_data["branch_code"].extend(branch_code_outw)
        clearing_outward_data["branch_name"].extend(branch_name_outw)
        clearing_outward_data["item_count"].extend(total_count_outw)
        clearing_outward_data["total_amount"].extend(total_amount_outw)
        clearing_outward_data["clearing_type"].extend(clearing_type_outw)
    else:
        return_inward_data["branch_code"].extend(branch_code_inw)
        return_inward_data["branch_name"].extend(branch_name_inw)
        return_inward_data["item_count"].extend(total_count_inw)
        return_inward_data["total_amount"].extend(total_amount_inw)
        return_inward_data["clearing_type"].extend(clearing_type_inw)
      
        return_outward_data["branch_code"].extend(branch_code_outw)
        return_outward_data["branch_name"].extend(branch_name_outw)
        return_outward_data["item_count"].extend(total_count_outw)
        return_outward_data["total_amount"].extend(total_amount_outw)
        return_outward_data["clearing_type"].extend(clearing_type_outw)


app=xw.App(visible=True)               #opens new workbook   #visible means changes are seen

app.books.add()   
app.books.add()

wb1=app.books('Book1')
wb2=app.books('Book2')
wb3=app.books('Book3')
               
wb1.sheets['Sheet1'].name='inward'     #renaming existing sheets
wb1.sheets.add('outward')              #adding new sheet with specified name

wb2.sheets['Sheet1'].name='inward'     
wb2.sheets.add('outward')   

wb3.sheets['Sheet1'].name='inward'     
wb3.sheets.add('outward') 

headers = ["Branch Code",
           "Branch Name",
           "Item Count",
           "Amount",
           "ClearingType" ]

# Write headers
wb1.sheets('inward').range("A1").value = headers
wb1.sheets('outward').range("A1").value = headers

wb2.sheets('inward').range("A1").value = headers
wb2.sheets('outward').range("A1").value = headers

wb3.sheets('inward').range("A1").value = headers
wb3.sheets('outward').range("A1").value = headers


wb1.sheets('inward').range('A2:E2').options(transpose=True).value=today_lodgment_inward_data["branch_code"],today_lodgment_inward_data["branch_name"],today_lodgment_inward_data["item_count"],today_lodgment_inward_data["total_amount"],today_lodgment_inward_data["clearing_type"]
wb1.sheets('outward').range('A2:E2').options(transpose=True).value=today_lodgment_inward_data["branch_code"],today_lodgment_outward_data["branch_name"],today_lodgment_outward_data["item_count"],today_lodgment_outward_data["total_amount"],today_lodgment_outward_data["clearing_type"]
wb1.sheets('inward').autofit()
wb1.sheets('outward').autofit()
wb2.sheets('inward').range('A2').options(transpose=True).value=clearing_inward_data["branch_code"],clearing_inward_data["branch_name"],clearing_inward_data["item_count"],clearing_inward_data["total_amount"],clearing_inward_data["clearing_type"]
wb2.sheets('outward').range('A2').options(transpose=True).value=clearing_outward_data["branch_code"],clearing_outward_data["branch_name"],clearing_outward_data["item_count"],clearing_outward_data["total_amount"],clearing_outward_data["clearing_type"]
wb2.sheets('inward').autofit()
wb2.sheets('outward').autofit()
wb3.sheets('inward').range('A2:E2').options(transpose=True).value=return_inward_data["branch_code"],return_inward_data["branch_name"],return_inward_data["item_count"],return_inward_data["total_amount"],return_inward_data["clearing_type"]  
wb3.sheets('outward').range('A2:E2').options(transpose=True).value=return_outward_data["branch_code"],return_outward_data["branch_name"],return_outward_data["item_count"],return_outward_data["total_amount"],return_outward_data["clearing_type"]    
wb3.sheets('inward').autofit()
wb3.sheets('outward').autofit()

#wb1.save("todays lodgment file.xlsx")
#wb2.save("clearing file.xlsx")
#wb3.save("return file.xlsx")

#wb1.close()
#wb2.close()
#wb3.close()

FileNotFoundError: [Errno 2] No such file or directory: 'BKTS10720010091124053218.PDF'

In [ ]:
"""Version2"""
""" This is an automation process designed for fetching and retrieving data from a certain directory 
and representing data in csv formats (containing particular information required) """


import xlwings as xw                            #version2
import pdfplumber
import re
from datetime  import datetime,date 
import os
#import glob
import pandas as pd


keys=['branch_code','branch_name','item_count','total_amount','clearing_type']
values=[[],[],[],[],[]]
today_lodgment_inward_data =pd.DataFrame( {k:v for (k,v) in zip(keys, values)})
today_lodgment_outward_data =pd.DataFrame ({k:v for (k,v) in zip(keys, values)})
clearing_inward_data =pd.DataFrame({k:v for (k,v) in zip(keys, values)})
clearing_outward_data =pd.DataFrame ({k:v for (k,v) in zip(keys,values)})
return_inward_data = pd.DataFrame({k:v for (k,v) in zip(keys, values)})
return_outward_data =pd.DataFrame({k:v for (k,v) in zip(keys, values)})

folder_path=(r"D:\tasks\task#5\NIFT_Files")
files=os.listdir(r"D:\tasks\task#5\NIFT_Files")
#files_series= (  glob.glob(os.path.join(folder_path, "BKTS107200*.pdf")) +
#                 glob.glob(os.path.join(folder_path, "BKTS107500*.pdf")) +
#                 glob.glob(os.path.join(folder_path, "BKTS107100*.pdf"))     )
files_series=[]
for file in files:
  if file.startswith("BKTS107200") or file.startswith("BKTS107500") or file.startswith("BKTS107100"):# and file.endswith(".pdf"):
    full_path = os.path.join(folder_path, file)
    files_series.append(full_path)    
#print(files_series)
#print(len(files_series))

for file in files:
  if file.lower().endswith(".pdf"):     #converts all letters in file name into lowercase and check if ends with pdf (to handle both pdf and PDF)
    file_full_path = os.path.join(folder_path, file)
    
    with pdfplumber.open(file_full_path) as pdf:
    
      text = "\n".join(page.extract_text() or "" for page in pdf.pages)
      date_match=re.findall(r"CLEARING AS ON\s*(\d{2}-\d{2}-\d{4})",text,re.MULTILINE)

      valid_date=None                           #validating date as real date
      if date_match!=[]:
        valid_date = datetime.strptime(date_match[0], '%d-%m-%Y').date()
      else:
        pass
      inward = re.findall(r"RECEIVED BY YOU.*?(.*?)DELIVERED BY YOU", text,re.DOTALL)
      inward = ' '.join(inward)
      branch_code_inw = re.findall(r"^\d{4}\s", inward,re.MULTILINE)
      branch_name_inw = re.findall(r"\d{4}\s+(.*?)\s+GROUP", inward,re.MULTILINE) 
      item_count_inw = re.findall(r"GROUP B\s*[:=]\s*(\d{1,2})\s",inward,re.MULTILINE)
      total_amount_inw = re.findall(r"GROUP B\s*[:=]\s\d{1,2}\s*([\d,.]+)", inward,re.MULTILINE)
      clearing_type_inw=["RECEIVED BY YOU"  for i in range(len(branch_code_inw)) ]
    
      values_inw=[branch_code_inw,branch_name_inw,item_count_inw,total_amount_inw,clearing_type_inw]
      df_inw=pd.DataFrame({k:v for (k,v) in zip(keys, values_inw)})

      outward = re.findall(r"DELIVERED BY YOU.*?(.*?)SUMMARY", text,re.DOTALL)
      outward = ' '.join(outward)

      branch_code_outw = re.findall(r"^\d{4}\s", outward,re.MULTILINE)
      branch_name_outw = re.findall(r"\d{4}\s+(.*?)\s+GROUP", outward,re.MULTILINE)
      item_count_outw = re.findall(r"GROUP B\s*[:=]\s*(\d{1,2})\s",outward,re.MULTILINE)
      total_amount_outw = re.findall(r"GROUP B\s*[:=]\s\d{1,2}\s*([\d,.]+)", outward,re.MULTILINE)
      clearing_type_outw=["DELIVERED BY YOU"   for i in range(len(branch_code_outw))]
    
      values_outw=[branch_code_outw,branch_name_outw,item_count_outw,total_amount_outw,clearing_type_outw]
      df_outw=pd.DataFrame({k:v for (k,v) in zip(keys, values_outw)})

      if valid_date==date.today():            #matching date with today's date
        today_lodgment_inward_data=pd.concat([today_lodgment_inward_data, df_inw], axis=0)
        today_lodgment_outward_data=pd.concat([today_lodgment_outward_data, df_outw], axis=0)
      else:pass

      if file_full_path in files_series:
        clearing_inward_data=pd.concat([clearing_inward_data, df_inw], axis=0)
        clearing_outward_data=pd.concat([clearing_outward_data,df_outw],axis=0)
      
      else:
        return_inward_data=pd.concat([return_inward_data, df_inw], axis=0)
        return_outward_data=pd.concat([return_outward_data, df_outw], axis=0)
#today_lodgment_inward_data.to_csv('today_lodgment_inward_data',index=False)
#today_lodgment_outward_data.to_csv('today_lodgment_outward_data',index=False)   #these two will be empty as per our given files
clearing_inward_data.to_csv('clearing_inward_data',index=False)
clearing_outward_data.to_csv('clearing_outward_data',index=False)
return_inward_data.to_csv('return_inward_data',index=False)
return_outward_data.to_csv('return_outward_data',index=False)

In [ ]:
"""Version3"""
""" This is automation process that reads pdf files from given folder,extracts infomation 
and saves them in separate csv files.

From each file it will extract following information:
i)_    clearing date,  
ii)_   branch_code,  
iii)_  branch_name,
iv)_   item_count,  
v)_    total_amount,   
vi)_   clearing_type.

Inside each file:
   If clearing type is-->  delivered,
      it will become inward data and 
   If clearing type is-->  received,
      it will go to outward data.
If file's clearing date is today's date,
      its inward and outward data will be stored in today lodgment file.
If file is from a specific files series,
      it will go to clearing file,
and rest of the files data will go to return file                             """


import xlwings as xw                            #version3
import pdfplumber
import re
from datetime  import datetime,date 
import os
import pandas as pd

from retriever_class import Retriever

keys=['branch_code','branch_name','item_count','total_amount','clearing_type']
values=[[],[],[],[],[]]
r1=Retriever(keys,values)

today_lodgment_inward_data =r1.create_dataframe(keys,values)
today_lodgment_outward_data =r1.create_dataframe(keys,values)
clearing_inward_data =r1.create_dataframe(keys,values)
clearing_outward_data =r1.create_dataframe(keys,values)
return_inward_data = r1.create_dataframe(keys,values)
return_outward_data =r1.create_dataframe(keys,values)

folder_path="D:\\tasks\\task5\\NIFT_Files"
#print("DOCS:::",r1.folder_files.__doc__)                     #method to call documentation of a function
files=r1.folder_files(folder_path)
#print(files)
series=r1.files_series(folder_path,"BKTS107200","BKTS107500","BKTS107100")
#print(series)

for file in files:
    inward=r1.inward_data(file,folder_path)        #dataframe
    #print(inward)
    #print(file)
    outward=r1.outward_data(file,folder_path)
    #print(outward)
    valid_date=r1.date_extraction(file,folder_path)
    #print(valid_date)

    if valid_date==date.today():            #matching date with today's date
       today_lodgment_inward_data=r1.join_dataframes(today_lodgment_inward_data, inward)
       today_lodgment_outward_data=r1.join_dataframes(today_lodgment_outward_data, outward)
    else:pass

    if file in series:
       clearing_inward_data=r1.join_dataframes(clearing_inward_data,inward)
       clearing_outward_data=r1.join_dataframes(clearing_outward_data,outward)
    else:
       return_inward_data=r1.join_dataframes(return_inward_data,inward)
       return_outward_data=r1.join_dataframes(return_outward_data,outward)

today_lodgment_inward_data.to_csv('today_lodgment_inward_data',index=False)
today_lodgment_outward_data.to_csv('today_lodgment_outward_data',index=False)
clearing_inward_data.to_csv('clearing_inward_data',index=False)
clearing_outward_data.to_csv('clearing_outward_data',index=False)
return_inward_data.to_csv('return_inward_data',index=False)
return_outward_data.to_csv('return_outward_data',index=False)



DOCS::: 
Retrieve a list of all files and directories within a specified folder.

Args:
    completefolderpath (str): The full path to the target directory.

Returns:
    list: A list containing the names of files and subdirectories
          in the given folder..

